In [1]:
import sys
from pathlib import Path
import os
import json

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

from Utils.DataUtils import build_ae_dataloaders
from Models.AutoEncoder import AutoEncoder

In [2]:
# Config
BATCH_SIZE = 256
LR = 1e-3
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# AE training, train only non-fraud
train_loader_ae, val_loader, test_loader, input_dim = build_ae_dataloaders(
    batch_size=BATCH_SIZE,
    mode="ae",
    train_only_nonfraud=True,
    return_labels=True
)

# Train fraud + non-fraud
train_loader_full, _, _, _ = build_ae_dataloaders(
    batch_size=BATCH_SIZE,
    mode="ae",
    train_only_nonfraud=False,
    return_labels=True
)

print("Input dim:", input_dim)

[INFO] Project root: C:\DeepLearning\DL_project
[INFO] Data dir: C:\DeepLearning\DL_project\data
[INFO] Loading train data from: C:\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: C:\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

[INFO] Train: (455902, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)
[INFO] Project root: C:\DeepLearning\DL_project
[INFO] Data dir: C:\DeepLearning\DL_project\data
[INFO] Loading train data from: C:\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: C:\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)
Input dim: 776


In [4]:
# AutoEncoder
model = AutoEncoder(
    input_dim=input_dim,
    latent_dim=256,              
    hidden_dims=[128, 64, 32],
    noise_std=0.01,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

In [5]:
# Training loop
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, _ in tqdm(train_loader_ae, desc=f"AE Epoch {epoch+1}"):
        x = x.to(DEVICE)

        x_hat, _ = model(x)
        loss = criterion(x_hat, x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader_ae):.6f}")

# Save AutoEncoder checkpoint
torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": input_dim,
    "latent_dim": 256,
    "hidden_dims": [128, 64, 32],
}, "checkpoints/autoencoder.pt")

print("AutoEncoder checkpoint saved")

AE Epoch 1: 100%|██████████████████████████████████████████████████████████████████| 1781/1781 [00:19<00:00, 89.17it/s]


Epoch 1 | Loss: 0.295398


AE Epoch 2: 100%|██████████████████████████████████████████████████████████████████| 1781/1781 [00:18<00:00, 95.65it/s]


Epoch 2 | Loss: 0.186951


AE Epoch 3: 100%|██████████████████████████████████████████████████████████████████| 1781/1781 [00:18<00:00, 95.53it/s]


...

Epoch 28 | Loss: 0.108475


AE Epoch 29: 100%|████████████████████████████████████████████████████████████████| 1781/1781 [00:16<00:00, 107.92it/s]


Epoch 29 | Loss: 0.140771


AE Epoch 30: 100%|████████████████████████████████████████████████████████████████| 1781/1781 [00:14<00:00, 120.12it/s]

Epoch 30 | Loss: 0.115384
AutoEncoder checkpoint saved
